In [ ]:
from openai import AzureOpenAI
import os
import json
import gradio as gr

/Users/rodrigo/Desktop/work_projects/AI_training/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Agentic Patterns

### Reflection pattern
The cells below show how the reflect pattern could be implemented from scratch using only the model providers APIs. In this case, we are using an endpoint we have deployed in Azure, hence we will be using the OpenAI SDK
1. We use `AzureOpenAI` library to access our endpoint.
2. Use the OpenAI-compatible chat completions API directly
2. `AzureOpenAI` client requires `api_key`, `azure_endpoint`, `api_version`
3. `completions` expects the name of the deployed resource as a parameter

In [2]:
client = AzureOpenAI(
    api_key = os.getenv("AZURE_OPENAI_KEY"),
    azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version = os.getenv("OPENAI_API_VERSION"))

In [3]:
WRITER_PROMPT = """You are a professional email writing assistant.
Given a rough draft (and optionally a critique of a previous version), rewrite the email to be:
- Professional and appropriate for a business setting
- Clear and concise
- Well-structured with a greeting, body, and sign-off

If a critique is provided, address every point raised in the critique.
Return ONLY the rewritten email, nothing else."""

CRITIQUE_PROMPT = """You are an expert email reviewer. Analyse the following email and provide:
1. A score from 1-10 (where 10 is perfect)
2. Specific feedback on:
   - Tone (is it professional and appropriate?)
   - Clarity (is it easy to understand?)
   - Length (is it concise without losing meaning?)
   - Structure (does it have a clear greeting, body, sign-off?)
3. Concrete suggestions for improvement

Format your response as:
SCORE: <number>
FEEDBACK:
<your detailed feedback>"""

In [4]:
messages = [{"role": "user", "content": "What is 2+2?"}]

response = client.chat.completions.create(
    messages=messages,
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
)

print(response.choices[0].message.content)

2 + 2 equals 4.


In [5]:
def write_email(user_input, critique : str | None = None):
    messages = [
        {"role": "system", "content": WRITER_PROMPT},
        {"role": "user", "content": f"Draft: {user_input}\nCritique: {critique}"}
    ]
    
    response = client.chat.completions.create(
        messages=messages,
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    )
    
    return response.choices[0].message.content


def critique_email(email : str):
    messages = [
        {"role": "system", "content": CRITIQUE_PROMPT},
        {"role": "user", "content": f"Email to critique: {email}"}
    ]
    
    response = client.chat.completions.create(
        messages=messages,
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    )
    
    return response.choices[0].message.content


#TODO: include pydantic model for validation reviewer.

In [6]:
MAX_ITERATIONS = 5
PASS_THRESHOLD = 8

my_email = "Hey Smon, can you send me the link to the deployed UI you lazy *****?"

for i in range(MAX_ITERATIONS):
    print(f"Iteration {i+1}:\n{my_email}\n")
    
    critique = critique_email(my_email)
    print(f"Critique:\n{critique}\n")
    
    score_line = critique.splitlines()[0]
    score = int(score_line.split(":")[1].strip())
        
    if score >= PASS_THRESHOLD:
        print("Email is good enough!")
        break
    
    my_email = write_email(my_email, critique)


Iteration 1:
Hey Smon, can you send me the link to the deployed UI you lazy *****?

Critique:
SCORE: 2

FEEDBACK:
1. **Tone**: 
   - The tone is unprofessional and inappropriate. Calling someone "lazy" and using a derogatory term ("*****") is disrespectful and could be perceived as offensive. While casual language might be acceptable in informal contexts among close colleagues or friends, this goes beyond casual and verges on insulting.
   - A professional tone is essential in business communication, and this email completely lacks professionalism.

2. **Clarity**:
   - The request to send the link to the deployed UI is clear in principle, but the phrasing detracts from the message. The offensive term and unprofessional tone make it harder for the recipient to focus on the actual request.
   - The message lacks context or additional information, such as a deadline or reason why the link is needed.

3. **Length**:
   - The email is very short, which is not inherently bad, but it sacrifi

Important: Structured outputs --> include section on this. this pattern is good for evaluating LLM response, and easy to implement.

### Tool Use Pattern
Frameworks such as pydantic AI, LangChain, Langgraph, etc. provide a convenient abstraction for implementing common Agentic AI capabilities, for example tool use. By using some decorators and injecting functions as parameters we can equip LLM with tools without worrying or thinking what is actually happening under the hood.
 

In [11]:
from tools import calculate, date_difference, convert_units

In [12]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression. Supports +, -, *, /, sqrt, pow, abs, round.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "The math expression to evaluate, e.g. '2 + 3 * 4'"}
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "date_difference",
            "description": "Calculate the number of days between two dates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "date1": {"type": "string", "description": "First date in YYYY-MM-DD format"},
                    "date2": {"type": "string", "description": "Second date in YYYY-MM-DD format"}
                },
                "required": ["date1", "date2"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_units",
            "description": "Convert a value between common units (km/miles, kg/lbs, celsius/fahrenheit).",
            "parameters": {
                "type": "object",
                "properties": {
                    "value": {"type": "number", "description": "The numeric value to convert"},
                    "from_unit": {"type": "string", "description": "The unit to convert from"},
                    "to_unit": {"type": "string", "description": "The unit to convert to"}
                },
                "required": ["value", "from_unit", "to_unit"]
            }
        }
    }
]

In [13]:
from tools import calculate, date_difference, convert_units

tool_registry = {
    "calculate": calculate,
    "date_difference": date_difference,
    "convert_units": convert_units
}

In [21]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name} with arguments: {tool_args}", flush=True)

        if tool_name == "calculate":
            result = tool_registry["calculate"](**tool_args)
        elif tool_name == "date_difference":
            result = tool_registry["date_difference"](**tool_args)
        elif tool_name == "convert_units":
            result = tool_registry["convert_units"](**tool_args)
        
        results.append({"role":"tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [22]:
SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions and call tools when needed to get information or perform calculations.
When you need to use a tool, call it with the appropriate arguments. Wait for the tool results before responding to the user. 
You can call multiple tools in a conversation if needed. You have access to the following tools:
1. calculate(expression): Evaluates a mathematical expression. Example: calculate("2 + 3 * 4")
2. date_difference(date1, date2): Calculates the number of days between two dates. Example: date_difference("2023-01-01", "2023-01-15")
3. convert_units(value, from_unit, to_unit): Converts a value between common units. Example: convert_units(10, "km", "miles")
"""

def chat(message, history):
    messages = [{"role" : "system", "content" : SYSTEM_PROMPT}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = client.chat.completions.create(
            messages=messages,
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            tools=tools,
            max_tokens=500
        )

        if response.choices[0].finish_reason == "tool_calls":
            message = response.choices[0].message
            results = handle_tool_calls(message.tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tool called: convert_units with arguments: {'value': 2.5, 'from_unit': 'miles', 'to_unit': 'meters'}
Tool called: date_difference with arguments: {'date1': '2026-10-06', 'date2': '2059-03-15'}


Under the hood:
What happens step by step
1) You send messages + tool definitions (tools=tools) to the model.
2) The model decides:
    -   either answer directly, or
    - request one or more tool calls.
3) If it requests tools, the response has finish_reason == "tool_calls" and includes structured tool call objects (message.tool_calls).
4) Your code loops those calls in handle_tool_calls(...):
    - gets tool name
    - gets tool arguments
    - executes the matching Python function from tool_registry
    - creates a "role": "tool" message with the result
5) You append:
    - the assistant message that requested tools
    - the tool result messages
6) You call the model again with updated messages.
7) Now the model has tool outputs, so it writes the final natural-language answer.


In [24]:
messages = [{"role": "user", "content": "What is the capital of France?"}]

resp1 = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    messages=messages
)

print(resp1.model_dump_json(indent=2))

{
  "id": "chatcmpl-Da0GHO99rNO7odzOYcnqngIwo5qvp",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "The capital of France is **Paris**.",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      },
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "protected_material_code": {
          "filtered": false,
          "detected": false
        },
        "protected_material_text": {
          "filtered": false,
          "detected": false
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
        }
    

In [25]:
messages = [{"role": "user", "content": "Say hello in one sentence."}]

resp2 = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    messages=messages,
    tools=tools
)

print(resp2.model_dump_json(indent=2))

{
  "id": "chatcmpl-Da0I3WltY6rUD7yhlNY1ktKbqeOHH",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Hello! I hope you're having a fantastic day!",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      },
      "content_filter_results": {
        "hate": {
          "filtered": false,
          "severity": "safe"
        },
        "protected_material_code": {
          "filtered": false,
          "detected": false
        },
        "protected_material_text": {
          "filtered": false,
          "detected": false
        },
        "self_harm": {
          "filtered": false,
          "severity": "safe"
        },
        "sexual": {
          "filtered": false,
          "severity": "safe"
        },
        "violence": {
          "filtered": false,
          "severity": "safe"
     

In [26]:
messages = [{"role": "user", "content": "What is 37 * 14 + 5?"}]

resp3 = client.chat.completions.create(
    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    messages=messages,
    tools=tools
)

print(resp3.model_dump_json(indent=2))

{
  "id": "chatcmpl-Da0IRNrQ4y8G5Q7cHq1s3xpmbkp2U",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": null,
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": [
          {
            "id": "call_BabxzurV3jLXQCEZF42Dnvzo",
            "function": {
              "arguments": "{\"expression\":\"37 * 14 + 5\"}",
              "name": "calculate"
            },
            "type": "function"
          }
        ]
      },
      "content_filter_results": {}
    }
  ],
  "created": 1777473319,
  "model": "gpt-4o-2024-11-20",
  "object": "chat.completion",
  "service_tier": "default",
  "system_fingerprint": "fp_af7f7349a4",
  "usage": {
    "completion_tokens": 20,
    "prompt_tokens": 195,
    "total_tokens": 215,
    "completion_tokens_details": {
      "accepted_prediction_tokens": 0,
      "audi

### Planning Agent

In [ ]:
PLANNER_PROMPT = """
You are a planning assistant.
Given a user request, produce a concise numbered plan (3-6 steps). Example format:
1. step one description
2. step two description
...

Return ONLY the numbered plan.
"""

STEP_EXECUTOR_PROMPT = """
You execute exactly one plan step at a time.
Use the user request and previous step outputs as context.
Return only the result for the current step.
"""

SYNTHESIZER_PROMPT = """
You are a final answer writer.
Given the user request and completed step outputs, produce the best final answer.
Be concise and clear.
"""

In [28]:
def planning_agent(user_request : str)->str:
    #Plan
    messages = [{"role" : "system", "content": PLANNER_PROMPT}] + [{"role": "user", "content": user_request}]
    plan_response = client.chat.completions.create(
        messages=messages,
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    )

    plan = plan_response.choices[0].message.content
    steps = []
    for line in plan.splitlines():
        line = line.strip()
        if line and line[0].isdigit():
            steps.append(line.split(".", 1)[1].strip())
    
    print("finished planning")
    print("steps to follow are: ", steps)
    
    step_outputs = []
    for i, step in enumerate(steps):
        step_messages = [{"role": "system", "content": STEP_EXECUTOR_PROMPT}] + [{"role": "user", "content": f"User request: {user_request}\nCurrent step: {step}\nPrevious step outputs: {step_outputs}"}]
        step_response = client.chat.completions.create(
            messages=step_messages,
            model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
        )
        step_output = step_response.choices[0].message.content
        step_outputs.append(step_output)
    

    step_results = json.dumps({f"Step {i+1}": {"description": step, "output": output} for i, (step, output) in enumerate(zip(steps, step_outputs))})
    synth_messages = [{"role": "system", "content": SYNTHESIZER_PROMPT}] + [{"role": "user", "content": f"User request: {user_request}\nCompleted step outputs:{step_results}"}]
    final_response = client.chat.completions.create(messages=synth_messages, 
                                                    model=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
                                                    )
    
    print(step_results)
    
    return final_response.choices[0].message.content

    


In [29]:
demo = planning_agent("create a three day beginner workout plan with rest recommendations")

finished planning
steps to follow are:  ['**Day 1 - Full Body Workout**:', '**Day 2 - Active Recovery**:', '**Day 3 - Upper and Core Focus**:']
{"Step 1": {"description": "**Day 1 - Full Body Workout**:", "output": "**Day 1 - Full Body Workout**:  \n- Warm-up: 5-10 minutes of light cardio (e.g., brisk walking, jogging, or jumping jacks)  \n- Squats: 3 sets of 10-12 repetitions  \n- Push-ups (modify to knee push-ups if needed): 3 sets of 8-10 repetitions  \n- Bent-over dumbbell rows (light weight): 3 sets of 10 repetitions per arm  \n- Plank: Hold for 20-30 seconds, repeat 2 times  \n- Cool-down: Stretching, focusing on legs, arms, and back for 5-10 minutes  \n\n**Rest Recommendations**: Stay hydrated, focus on deep breathing post-workout, and avoid strenuous activities to allow your muscles to recover."}, "Step 2": {"description": "**Day 2 - Active Recovery**:", "output": "**Day 2 - Active Recovery**:  \n- Light Cardio: 15-20 minutes of walking, cycling, or swimming at a moderate pace 

In [30]:
print(demo)

**Three-Day Beginner Workout Plan with Rest Recommendations:**

**Day 1 - Full Body Workout**:  
- Warm-up: 5-10 minutes of light cardio (e.g., brisk walking, jogging, or jumping jacks)  
- Squats: 3 sets of 10-12 repetitions  
- Push-ups (modify to knee push-ups if needed): 3 sets of 8-10 repetitions  
- Bent-over dumbbell rows (light weight): 3 sets of 10 repetitions per arm  
- Plank: Hold for 20-30 seconds, repeat 2 times  
- Cool-down: Stretching, focusing on legs, arms, and back for 5-10 minutes

**Rest Recommendations**: Stay hydrated, focus on deep breathing, and avoid strenuous activities to allow muscle recovery.

---

**Day 2 - Active Recovery**:  
- Light Cardio: 15-20 minutes of walking, cycling, or swimming at a moderate pace  
- Mobility Work: Perform dynamic stretches such as arm circles, leg swings, and cat-cow stretches (2-3 repetitions per movement)  
- Yoga or Stretching: Gentle poses like child’s pose, downward dog, and seated forward fold for 10-15 minutes  
- Coo

### MultiAgent Pattern